# 06 Hyperparameter Optimization

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp017_hparam_coarse.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp017_hparam_fine.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_fine.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp017_hparam_coarse_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp017_hparam_final_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_final_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '30cf336abe6e762a443ae64eb0f10f64e109639e'}, '05_model_optimize/MLP/Normal/scripts/exp017_hparam_fine.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b0bcd0b435e89d2f8eb91498b94d8e6e6e122d7e'}, '05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'c6df0646762020e0e8ee39a1f24a5e669c2e110b'}, '05_model_optimize/MLP/Normal/scripts/exp017_hparam_final_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ee2e812dbdee8f2598c59a5f53e25bde98b197a0'}}

## exp017_hparam_coarse

원본: [05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse.py)  
Git blob: `30cf336abe6e762a443ae64eb0f10f64e109639e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP017 stage 1: 80-14 fixed preprocessing. Tune MLP only using Official Train/Val.

Team handwriting, Original Test and professor Test are NEVER read. Generate exact
EXP016 80-14 deterministic augmented images, independently of MLP seed.

CLI 인자 (정적 추출): `--seed`, `--part`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP017 stage 1: 80-14 fixed preprocessing. Tune MLP only using Official Train/Val.

Team handwriting, Original Test and professor Test are NEVER read. Generate exact
EXP016 80-14 deterministic augmented images, independently of MLP seed.
"""
from __future__ import annotations
import argparse, warnings, time, json, csv
from pathlib import Path
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from exp006_size_com_comparison import ROOT, LABELS, PixelScaler255, load_official, process, image_geometry
from exp011_012_causal_probes import fixed_scale
from exp016_high_fraction_grid import nested_subset, scale_base, stroke_jitter
from exp014_balanced_augmentation import balanced_indices

SEEDS=(42,43,44)
SCALES=(.8,1.,1.2)
HIDDEN=((100,),(128,),(192,),(256,),(128,64),(256,128))
MORE=((128,),(192,),(128,64),(256,128))
CONF=(
    [(h,a,.001) for h in HIDDEN for a in (.0001,.001,.01)]
    +[(h,.001,lr) for h in MORE for lr in (.0005,.002)]
)
assert len(CONF)==26
def config_name(h,a,lr):
    return "h"+"x".join(map(str,h))+"_a"+f"{a:.4g}".replace(".","p").replace("-","m")+"_lr"+f"{lr:.4g}".replace(".","p").replace("-","m")
def load_data():
    official=load_official(ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz")
    Xraw=official["X_train"]
    Y=np.asarray(official["y_train"],dtype=np.int32)
    Xv=official["X_val"]
    Yv=np.asarray(official["y_val"],dtype=np.int32)
    assert Xraw.shape==(76800,784) and Xv.shape==(19200,784)
    assert set(np.unique(Y))==set(range(16)) and set(np.unique(Yv))==set(range(16))
    target=int(np.median(image_geometry(Xraw)["long_side"]))
    assert target==20
    train=process(Xraw,"com",target)
    ids,targets=nested_subset(Y,.8)
    ids_half,_=balanced_indices(Y,seed=20261116)
    resized=scale_base(Xraw,ids,targets)
    changed,stats=stroke_jitter(resized,.14,ids,ids_half)
    train[ids]=process(changed,"com",target)
    assert len(ids)==61440 and train.shape==(76800,784),len(ids)
    val={s:process(Xv if s==1. else fixed_scale(Xv,s),"com",target) for s in SCALES}
    assert all(v.shape==(19200,784) for v in val.values())
    return train,Y,val,Yv,official["sha256"],len(ids),stats

def measure(model, val, truth, seed, name, config):
    pred={}; proba={}
    for scale in SCALES:
        probabilities=np.asarray(model.predict_proba(val[scale]),dtype=np.float32)
        assert probabilities.shape==(len(truth),16)
        pred[scale]=probabilities.argmax(axis=1).astype(np.uint8)
        proba[scale]=probabilities[:,10:].sum(axis=1)
    base_family=proba[1.]>=.5
    rows=[];cls=[]
    for scale in SCALES:
        for label,mask in [("all",np.ones(len(truth),dtype=bool)),("digits",truth<10),("letters",truth>=10)]+[
            ("class_"+sym,truth==j) for j,sym in enumerate(LABELS)]:
            x=proba[scale][mask];base=proba[1.][mask]
            rows.append({"seed":seed,"variant":name,"hidden":str(config[0]),"alpha":config[1],
                "lr":config[2],"scale":scale,"group":label,"n":int(mask.sum()),
                "accuracy":float(np.mean(pred[scale][mask]==truth[mask])),
                "family_flip":float(np.mean((x>=.5)!=base_family[mask])),
                "abs_delta_p_letter":float(np.mean(np.abs(x-base))),
                "signed_delta_p_letter":float(np.mean(x-base))})
    return rows,{"p_letter":np.stack([proba[s] for s in SCALES]).astype(np.float32),
                 "pred":np.stack([pred[s] for s in SCALES]).astype(np.uint8),
                 "labels":truth.astype(np.uint8),"scales":np.array(SCALES,dtype=np.float32)}
def save(path,rows):
    assert rows
    with Path(path).open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)

def run(seed,part,out):
    assert seed in SEEDS and part in (0,1,2)
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    started=time.monotonic()
    X,Y,val,yv,digest,naug,stroke_stats=load_data()
    conf=[(i,*c) for i,c in enumerate(CONF) if i%3==part]
    measurements=[];records=[]
    for index,h,a,lr in conf:
        name=config_name(h,a,lr)
        begin=time.monotonic()
        clf=Pipeline([("scale_0_1",PixelScaler255()),
            ("model",MLPClassifier(hidden_layer_sizes=h,alpha=a,learning_rate_init=lr,
                activation="relu",solver="adam",max_iter=200,early_stopping=False,
                random_state=seed))])
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter("always")
            clf.fit(X,Y)
        path=out/(name+".joblib")
        joblib.dump(clf,path,compress=3)
        rows,detail=measure(clf,val,yv,seed,name,(h,a,lr))
        measurements+=rows
        np.savez_compressed(out/(name+"_officialval_response.npz"),**detail)
        base=next(r for r in rows if r["group"]=="all" and r["scale"]==1.)
        records.append({"seed":seed,"index":index,"variant":name,
          "hidden":str(h),"alpha":a,"learning_rate_init":lr,
          "activation":"relu","solver":"adam","max_iter":200,"early_stopping":False,
          "n_iter":clf.named_steps["model"].n_iter_,
          "official_val_accuracy":base["accuracy"],
          "fit_seconds":time.monotonic()-begin,
          "warning":" | ".join(str(w.message) for w in ws),
          "train_n":len(Y),"augmented_n":naug,
          "stroke_modified":stroke_stats["modified"],"stroke_rejected":stroke_stats["rejected"],
          "official_source_sha256":digest})
        print("EXP017_COARSE",seed,part,index,name,"acc",round(base["accuracy"],5),
              "iters",clf.named_steps["model"].n_iter_,flush=True)
        del clf
    save(out/"metrics.csv",measurements)
    save(out/"training_records.csv",records)
    manifest={"experiment":"EXP017","stage":"coarse","seed":seed,"part":part,
       "config_count":len(conf),"configs":[r["variant"] for r in records],
       "training_rows":76800,"official_val_rows":19200,"augmented_rows":naug,
       "original_official_sha256":digest,"size_aug_ratio":.8,
       "stroke_blend":.14,"preprocessing":"size_stroke_com_train__com_val",
       "team_handwriting_loaded":False,"handmade_used":False,
       "original_test_used":False,"professor_test_used":False,
       "runtime_seconds":time.monotonic()-started}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP017_COARSE_COMPLETE",seed,part,flush=True)

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--seed",type=int,required=True,choices=SEEDS)
    p.add_argument("--part",type=int,required=True,choices=(0,1,2))
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    run(a.seed,a.part,a.out)


## exp017_hparam_fine

원본: [05_model_optimize/MLP/Normal/scripts/exp017_hparam_fine.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_fine.py)  
Git blob: `b0bcd0b435e89d2f8eb91498b94d8e6e6e122d7e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP017 fine stage; dynamically selected architectures from frozen coarse study.

No Handmade file accesses. Official Train 76800-only fitting; OfficialVal evaluation.

CLI 인자 (정적 추출): `--seed`, `--slot`, `--coarse-json`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_hparam_fine.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP017 fine stage; dynamically selected architectures from frozen coarse study.

No Handmade file accesses. Official Train 76800-only fitting; OfficialVal evaluation.
"""
from __future__ import annotations
import argparse,json,warnings,time
from pathlib import Path
import joblib
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from exp006_size_com_comparison import PixelScaler255
from exp017_hparam_coarse import load_data,measure,save,SEEDS

KINDS=("alpha_down","alpha_up","lr_down","lr_up","early_stopping")
def configs(coarse_json):
    doc=json.loads(Path(coarse_json).read_text(encoding="utf-8"))
    assert doc["stage"]=="coarse" and doc["experiment"]=="EXP017"
    selected=doc["selected_4_architectures"]
    assert len(selected)==4 and len({tuple(x["hidden"]) for x in selected})==4
    allc=[]
    for slot,v in enumerate(selected):
        h=tuple(v["hidden"]);a=float(v["alpha"]);lr=float(v["lr"])
        for kind in KINDS:
            aa=a;ll=lr;early=False
            if kind=="alpha_down":aa=max(1e-6,a*.3)
            if kind=="alpha_up":aa=min(.1,a*3)
            if kind=="lr_down":ll=lr*.7
            if kind=="lr_up":ll=lr*1.4
            if kind=="early_stopping":early=True
            name=f"slot{slot+1}_{kind}"
            allc.append({"variant":name,"base_variant":v["variant"],
              "slot":slot,"kind":kind,"hidden":h,"alpha":aa,
              "learning_rate_init":ll,"early_stopping":early,
              "max_iter":260 if early else 200,
              "n_iter_no_change":15 if early else 10,
              "validation_fraction":.1 if early else .1})
    assert len(allc)==20 and len({x["variant"] for x in allc})==20
    return allc
def run(seed,slot,coarse_json,out):
    assert seed in SEEDS and slot in range(4)
    selected=[c for c in configs(coarse_json) if c["slot"]==slot]
    assert len(selected)==5
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    started=time.monotonic()
    X,Y,val,yv,digest,naug,stats=load_data()
    records=[];metrics=[]
    for config in selected:
        name=config["variant"]
        args={k:config[k] for k in ("hidden","alpha","learning_rate_init","early_stopping","max_iter","n_iter_no_change","validation_fraction") if k!="hidden"}
        args["hidden_layer_sizes"]=config["hidden"]
        args.update({"activation":"relu","solver":"adam","random_state":seed})
        model=Pipeline([("scale_0_1",PixelScaler255()),("model",MLPClassifier(**args))])
        begin=time.monotonic()
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            model.fit(X,Y)
        joblib.dump(model,out/(name+".joblib"),compress=3)
        rows,response=measure(model,val,yv,seed,name,(config["hidden"],config["alpha"],config["learning_rate_init"]))
        metrics+=rows
        import numpy as np
        np.savez_compressed(out/(name+"_officialval_response.npz"),**response)
        accuracy=next(x["accuracy"] for x in rows if x["scale"]==1. and x["group"]=="all")
        records.append({**{k:(list(v) if isinstance(v,tuple) else v) for k,v in config.items()},
                        "seed":seed,"n_train_source":len(Y),"n_augment":naug,
                        "original_official_sha256":digest,
                        "stroke_modified":stats["modified"],"stroke_rejected":stats["rejected"],
                        "n_iter":model.named_steps["model"].n_iter_,
                        "official_val_accuracy":accuracy,
                        "fit_seconds":time.monotonic()-begin,
                        "warnings":" | ".join(str(i.message) for i in w)})
        print("EXP017_FINE",seed,slot,name,"acc",round(accuracy,6),
              "iter",model.named_steps["model"].n_iter_,flush=True)
        del model
    save(out/"training_records.csv",records)
    save(out/"metrics.csv",metrics)
    manifest={"experiment":"EXP017","stage":"fine","seed":seed,"slot":slot,
       "base_configs":[c["base_variant"] for c in selected],
       "configs":[x["variant"] for x in selected],
       "training_rows_source":76800,"official_val_rows":19200,"augmented_rows":naug,
       "official_source_sha256":digest,"preprocessing":"80pct_size_14pct_stroke_com",
       "team_handwriting_loaded":False,"handmade_used":False,
       "official_test_used":False,"professor_test_used":False,
       "duration_seconds":time.monotonic()-started}
    (out/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("EXP017_FINE_COMPLETE",seed,slot,flush=True)
if __name__=="__main__":
    ap=argparse.ArgumentParser()
    ap.add_argument("--seed",type=int,required=True,choices=SEEDS)
    ap.add_argument("--slot",type=int,required=True,choices=(0,1,2,3))
    ap.add_argument("--coarse-json",required=True,type=Path)
    ap.add_argument("--out",required=True,type=Path)
    args=ap.parse_args()
    run(args.seed,args.slot,args.coarse_json,args.out)


## exp017_hparam_coarse_aggregate

원본: [05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse_aggregate.py)  
Git blob: `c6df0646762020e0e8ee39a1f24a5e669c2e110b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Stage-1 EXP017 predeclared 3-seed gate-based OfficialVal aggregation.

No training and no handwriting loading. Returns up to four DIFFERENT hidden
layer architectures for stage-2 fine hyperparameter search.

CLI 인자 (정적 추출): `--inputs`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_hparam_coarse_aggregate.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Stage-1 EXP017 predeclared 3-seed gate-based OfficialVal aggregation.

No training and no handwriting loading. Returns up to four DIFFERENT hidden
layer architectures for stage-2 fine hyperparameter search.
"""
from __future__ import annotations
import argparse,csv,json,statistics
from pathlib import Path
from collections import defaultdict
from exp017_hparam_coarse import CONF,SEEDS,config_name
GATE={"min_official_val":.9431076388888888, # frozen 80-14 0.9481076389 - 0.005
      "max_focus_flip":.20,"max_any16_flip":.25,"max_single_seed_focus_flip":.30,
      "max_digits12_flip":.10,"max_letters08_flip":.10,
      "max_digits12_pdrift":.12,"max_letters08_pdrift":.12,
      "min_digit_acc":.85,"min_letter_acc":.85}
FOCUS=("8","B","D")
CLASSES=tuple("0123456789ABCDEF")
SCALES=(.8,1.,1.2)
def read_csv(path):
    with Path(path).open("r",encoding="utf-8",newline="") as f:return list(csv.DictReader(f))
def write_csv(path,rows):
    assert rows
    with Path(path).open("w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def avg(values):return sum(values)/len(values)
def run(inputs,out):
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    config={config_name(h,a,lr):{"hidden":list(h),"alpha":a,"lr":lr}
            for h,a,lr in CONF}
    assert len(config)==26
    raw=defaultdict(list);train=defaultdict(list);hashes=set()
    for seed in SEEDS:
      for part in (0,1,2):
        path=Path(inputs)/f"exp017-coarse-s{seed}-p{part}"
        manifest=json.loads((path/"manifest.json").read_text(encoding="utf-8"))
        assert manifest["experiment"]=="EXP017" and manifest["seed"]==seed and manifest["part"]==part
        assert manifest["training_rows"]==76800 and manifest["official_val_rows"]==19200
        assert manifest["augmented_rows"]==61440
        assert manifest["size_aug_ratio"]==.8 and manifest["stroke_blend"]==.14
        assert not any(manifest[k] for k in ("team_handwriting_loaded","handmade_used","original_test_used","professor_test_used"))
        hashes.add(manifest["original_official_sha256"])
        for r in read_csv(path/"training_records.csv"):
            name=r["variant"];key=(name,seed)
            assert name in config and not train[key]
            train[key].append(r)
        for r in read_csv(path/"metrics.csv"):
            key=(r["variant"],seed,r["group"],float(r["scale"]))
            assert not raw[key],key
            raw[key].append(float(r["accuracy"]))
            raw[key].append(float(r["family_flip"]))
            raw[key].append(float(r["abs_delta_p_letter"]))
    assert len(hashes)==1 and len(train)==26*3
    assert all((name,seed) in train for name in config for seed in SEEDS)
    def vals(name,group,scale,idx):
        return [raw[name,seed,group,scale][idx] for seed in SEEDS]
    ranked=[]
    for name,v in config.items():
        v_acc=vals(name,"all",1.,0)
        d_acc=vals(name,"digits",1.,0); l_acc=vals(name,"letters",1.,0)
        dflip=vals(name,"digits",1.2,1); lflip=vals(name,"letters",.8,1)
        dp=vals(name,"digits",1.2,2);lp=vals(name,"letters",.8,2)
        focal=max((avg(vals(name,"class_"+c,scale,1)),c,scale) for c in FOCUS for scale in (.8,1.2))
        all16=max((avg(vals(name,"class_"+c,scale,1)),c,scale) for c in CLASSES for scale in (.8,1.2))
        focus_seed=max(raw[name,seed,"class_"+c,scale][1] for seed in SEEDS
                       for c in FOCUS for scale in (.8,1.2))
        failed=[]
        if avg(v_acc)<GATE["min_official_val"]:failed.append("val_below_anchor_minus_0p5pp")
        if focal[0]>GATE["max_focus_flip"]:failed.append("B8D_above_20pct")
        if all16[0]>GATE["max_any16_flip"]:failed.append("anyclass_above_25pct")
        if focus_seed>GATE["max_single_seed_focus_flip"]:failed.append("B8D_seed_above_30pct")
        if avg(dflip)>GATE["max_digits12_flip"]:failed.append("digits_1p2_flip_above_10pct")
        if avg(lflip)>GATE["max_letters08_flip"]:failed.append("letters_0p8_flip_above_10pct")
        if avg(dp)>GATE["max_digits12_pdrift"]:failed.append("digits_probability_drift")
        if avg(lp)>GATE["max_letters08_pdrift"]:failed.append("letters_probability_drift")
        if avg(d_acc)<GATE["min_digit_acc"]:failed.append("digit_accuracy_below_85pct")
        if avg(l_acc)<GATE["min_letter_acc"]:failed.append("letter_accuracy_below_85pct")
        row={"variant":name,"hidden":str(v["hidden"]),"alpha":v["alpha"],"learning_rate_init":v["lr"],
          "pass_all_gates":not failed,"failed_gates":" | ".join(failed),"n_failed_gates":len(failed),
          "officialval_acc_mean":avg(v_acc),"officialval_acc_sd":statistics.stdev(v_acc),
          "officialval_seed42":v_acc[0],"officialval_seed43":v_acc[1],"officialval_seed44":v_acc[2],
          "officialval_digits_mean":avg(d_acc),"officialval_letters_mean":avg(l_acc),
          "officialval_scale08_acc":avg(vals(name,"all",.8,0)),
          "officialval_scale12_acc":avg(vals(name,"all",1.2,0)),
          "digits12_flip":avg(dflip),"letters08_flip":avg(lflip),
          "digits12_pdrift":avg(dp),"letters08_pdrift":avg(lp),
          "B8D_max_flip":focal[0],"B8D_worst_class":focal[1],"B8D_worst_scale":focal[2],
          "max16class_flip":all16[0],"max16_worst_class":all16[1],
          "max16_worst_scale":all16[2],"max_single_seed_B8D_flip":focus_seed,
          "mean_n_iter":avg([float(train[name,seed][0]["n_iter"]) for seed in SEEDS])}
        ranked.append(row)
    eligible=sorted([x for x in ranked if x["pass_all_gates"]],
        key=lambda x:(-x["officialval_acc_mean"],x["B8D_max_flip"],x["max16class_flip"]))
    rejected=sorted([x for x in ranked if not x["pass_all_gates"]],
        key=lambda x:(x["n_failed_gates"],x["B8D_max_flip"],-x["officialval_acc_mean"]))
    ordered=eligible+rejected
    selected=[];seen=set()
    for row in ordered:
        if row["hidden"] in seen:continue
        selected.append(row)
        seen.add(row["hidden"])
        if len(selected)==4:break
    assert len(selected)==4
    names={r["variant"] for r in selected}
    for j,row in enumerate(ordered):
        row["overall_rank"]=j+1
        row["stage2_selected"]=row["variant"] in names
    write_csv(out/"all_26_configs_3seed_gates.csv",ordered)
    metadata={"experiment":"EXP017","stage":"coarse","models":78,"configurations":26,
      "official_train_n":76800,"official_val_n":19200,
      "team_handwriting_used":False,"original_test_used":False,"professor_test_used":False,
      "official_source_sha256":list(hashes)[0],"gate_predeclared":GATE,
      "passing_configs":len(eligible),"selected_4_architectures":[
           {"variant":x["variant"],"hidden":config[x["variant"]]["hidden"],
            "alpha":config[x["variant"]]["alpha"],"lr":config[x["variant"]]["lr"],
            "passes_all_gates":x["pass_all_gates"]} for x in selected]}
    (out/"metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
    summary=["# EXP유017 — OfficialVal 전용 26구성 MLP 3-Seed 탐색",
       "",f"총 **78개 모델**; 안전 Gate 통과 구성: **{len(eligible)}/26**.",
       "Hand-made/팀 수집 손글씨를 읽거나 모델 선택에 쓰지 않음.",
       "",
       "| 조건 | 합격 | OfficialVal | 숫자×1.2 Flip | 문자×0.8 Flip | B8D 최악 | 16클래스 최악 |",
       "|---|---|---:|---:|---:|---:|---:|"]
    for r in ordered:
        summary.append(f"| {r['variant']} | {'O' if r['pass_all_gates'] else 'X'} | "+
          " | ".join(f"{r[k]*100:.2f}%" for k in 
              ("officialval_acc_mean","digits12_flip","letters08_flip","B8D_max_flip","max16class_flip"))+" |")
    summary+=["","## 다음 단계 후보(서로 다른 4개 구조)"]+[
       f"- {s['variant']} — {'Gate 통과' if s['passes_all_gates'] else 'Gate 미달·대체 후보'}"
       for s in metadata["selected_4_architectures"]]
    (out/"SUMMARY.md").write_text("\n".join(summary)+"\n",encoding="utf-8")
    print("EXP017_COARSE_AGGREGATE_OK",len(eligible),"stage2",metadata["selected_4_architectures"],flush=True)

if __name__=="__main__":
    a=argparse.ArgumentParser()
    a.add_argument("--inputs",type=Path,required=True)
    a.add_argument("--out",type=Path,required=True)
    x=a.parse_args();run(x.inputs,x.out)


## exp017_hparam_final_aggregate

원본: [05_model_optimize/MLP/Normal/scripts/exp017_hparam_final_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp017_hparam_final_aggregate.py)  
Git blob: `ee2e812dbdee8f2598c59a5f53e25bde98b197a0`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP017 final coarse+fine joint ranking; OfficialVal-only, no handwritten data.

Original Test/professor Test unaccessed. No subsequent tuning against Hand-made.

CLI 인자 (정적 추출): `--coarse-inputs`, `--fine-inputs`, `--coarse-json`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp017_hparam_final_aggregate.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP017 final coarse+fine joint ranking; OfficialVal-only, no handwritten data.

Original Test/professor Test unaccessed. No subsequent tuning against Hand-made.
"""
from __future__ import annotations
import argparse,csv,json,statistics
from pathlib import Path
from exp017_hparam_coarse_aggregate import GATE,FOCUS,CLASSES,avg
from exp017_hparam_coarse import CONF,config_name,SEEDS
from exp017_hparam_fine import configs as fine_configs

def read(path):
    with Path(path).open("r",encoding="utf-8",newline="") as f:
        return list(csv.DictReader(f))
def save(path,rows):
    with Path(path).open("w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
def run(coarse_inputs,fine_inputs,coarse_json,out):
    out=Path(out)
    if out.exists():raise FileExistsError(out)
    out.mkdir(parents=True)
    cset={config_name(h,a,lr):{"hidden":list(h),"alpha":a,"lr":lr,
        "early_stopping":False,"source":"coarse"} for h,a,lr in CONF}
    fset={x["variant"]:{**x,"source":"fine"} for x in fine_configs(coarse_json)}
    names={**cset,**fset}
    assert len(names)==46 and len(cset)==26 and len(fset)==20
    data={};paths={};fit={};sha=set()
    for seed in SEEDS:
      for stage,root,splits in (("coarse",coarse_inputs,3),("fine",fine_inputs,4)):
        for j in range(splits):
            tag=f"exp017-coarse-s{seed}-p{j}" if stage=="coarse" else f"exp017-fine-s{seed}-slot{j}"
            f=Path(root)/tag
            man=json.loads((f/"manifest.json").read_text(encoding="utf-8"))
            assert man["experiment"]=="EXP017" and man["stage"]==stage and man["seed"]==seed
            assert not any(man[k] for k in ("team_handwriting_loaded","handmade_used","original_test_used","professor_test_used"))
            assert man["official_val_rows"]==19200 and man["augmented_rows"]==61440
            assert (man["training_rows"] if stage=="coarse" else man["training_rows_source"])==76800
            sha.add(man["original_official_sha256"] if stage=="coarse" else man["official_source_sha256"])
            for record in read(f/"training_records.csv"):
                name=record["variant"]
                assert name in names and names[name]["source"]==stage
                assert (name,seed) not in fit
                fit[name,seed]=record
                paths[name,seed]=str(f)
            for r in read(f/"metrics.csv"):
                name=r["variant"]
                key=(name,seed,r["group"],float(r["scale"]))
                assert key not in data
                data[key]=(float(r["accuracy"]),float(r["family_flip"]),float(r["abs_delta_p_letter"]))
    assert len(sha)==1 and len(fit)==46*3,(sha,len(fit))
    def vals(name,grp,scale,index):return [data[name,s,grp,scale][index] for s in SEEDS]
    rows=[]
    for name,c in names.items():
        val=vals(name,"all",1.,0)
        digit=vals(name,"digits",1.,0);letter=vals(name,"letters",1.,0)
        d12=vals(name,"digits",1.2,1);l08=vals(name,"letters",.8,1)
        dp=vals(name,"digits",1.2,2);lp=vals(name,"letters",.8,2)
        focus=max((avg(vals(name,"class_"+cls,sc,1)),cls,sc) for cls in FOCUS for sc in (.8,1.2))
        all16=max((avg(vals(name,"class_"+cls,sc,1)),cls,sc) for cls in CLASSES for sc in (.8,1.2))
        seedmax=max(data[name,seed,"class_"+cls,sc][1] for seed in SEEDS for cls in FOCUS for sc in (.8,1.2))
        failed=[]
        if avg(val)<GATE["min_official_val"]:failed.append("val_below_anchor_minus_0p5pp")
        if focus[0]>GATE["max_focus_flip"]:failed.append("B8D_above_20pct")
        if all16[0]>GATE["max_any16_flip"]:failed.append("anyclass_above_25pct")
        if seedmax>GATE["max_single_seed_focus_flip"]:failed.append("B8D_seed_above_30pct")
        if avg(d12)>GATE["max_digits12_flip"]:failed.append("digits_1p2_flip_above_10pct")
        if avg(l08)>GATE["max_letters08_flip"]:failed.append("letters_0p8_flip_above_10pct")
        if avg(dp)>GATE["max_digits12_pdrift"]:failed.append("digits_probability_drift")
        if avg(lp)>GATE["max_letters08_pdrift"]:failed.append("letters_probability_drift")
        if avg(digit)<GATE["min_digit_acc"]:failed.append("digit_accuracy_below_85pct")
        if avg(letter)<GATE["min_letter_acc"]:failed.append("letter_accuracy_below_85pct")
        rows.append({"variant":name,"source":c["source"],
          "hidden":str(c["hidden"]),"alpha":c["alpha"],
          "learning_rate_init":c["lr"] if c["source"]=="coarse" else c["learning_rate_init"],
          "early_stopping":c["early_stopping"],
          "passes_all_gates":not failed,"fail_count":len(failed),"failed_gates":" | ".join(failed),
          "OfficialVal_mean":avg(val),"OfficialVal_stdev":statistics.stdev(val),
          "OfficialVal_seed42":val[0],"OfficialVal_seed43":val[1],"OfficialVal_seed44":val[2],
          "Val_digit_accuracy":avg(digit),"Val_letter_accuracy":avg(letter),
          "Val_scale08_accuracy":avg(vals(name,"all",.8,0)),
          "Val_scale12_accuracy":avg(vals(name,"all",1.2,0)),
          "digits_1p2_flip":avg(d12),"letters_0p8_flip":avg(l08),
          "digits_1p2_pdrift":avg(dp),"letters_0p8_pdrift":avg(lp),
          "B8D_max_flip":focus[0],"B8D_class":focus[1],"B8D_scale":focus[2],
          "all16_max_flip":all16[0],"all16_class":all16[1],
          "all16_scale":all16[2],"B8D_max_one_seed":seedmax,
          "training_iterations_mean":avg([float(fit[name,s]["n_iter"]) for s in SEEDS])})
    eligible=sorted([x for x in rows if x["passes_all_gates"]],
          key=lambda x:(-x["OfficialVal_mean"],x["B8D_max_flip"],x["all16_max_flip"]))
    excluded=sorted([x for x in rows if not x["passes_all_gates"]],
          key=lambda x:(x["fail_count"],x["B8D_max_flip"],-x["OfficialVal_mean"]))
    ranked=eligible+excluded
    for j,r in enumerate(ranked):
        r["gate_rank"]=j+1 if j<len(eligible) else ""
        r["status"]="ELIGIBLE" if j<len(eligible) else "NOT_ELIGIBLE"
    save(out/"all_46_configs_3seed_final_ranking.csv",ranked)
    def meta(r):
        return None if r is None else {"variant":r["variant"],"source":r["source"],
          "hidden":r["hidden"],"alpha":r["alpha"],"learning_rate_init":r["learning_rate_init"],
          "early_stopping":r["early_stopping"],"OfficialVal_mean":r["OfficialVal_mean"],
          "B8D_max_flip":r["B8D_max_flip"],"all16_max_flip":r["all16_max_flip"]}
    leader=eligible[0] if eligible else None
    runner=eligible[1] if len(eligible)>1 else None
    fallback=excluded[0] if excluded else None
    md={"experiment":"EXP017","stage":"joint_final","n_coarse":26,"n_fine":20,
        "n_models":46*3,"n_total_conditions":46,
        "seed_count":3,"gates":GATE,"official_val_source_sha256":list(sha)[0],
        "all_gates_passing":len(eligible),"selected_frozen_candidate":meta(leader),
        "runner_up":meta(runner),"best_noneligible_diagnostic":meta(fallback),
        "team_handwriting_evaluated":False,"original_test_evaluated":False,
        "professor_test_evaluated":False,
        "stage1_frozen_80_14_anchor_officialval":.9481076388888888,
        "stage1_frozen_80_14_anchor_B8D_max_officialval":.25556}
    (out/"metadata.json").write_text(json.dumps(md,ensure_ascii=False,indent=2),encoding="utf-8")
    summary=["# EXP유017 MLP HP — OfficialVal-only 통합 최종 결과",
      "",f"1차 26개 + 2차 20개 = **46조합×3 Seed=138개 학습 완료**. 통과 조합 **{len(eligible)}개**.",
      "**팀 손글씨/기존 Hand-made/Original Test/교수님 Test는 튜닝에 일절 사용하지 않음.**",
      "기존 80-14 OfficialVal 94.8108%, B·D·8 최악 확대·축소 판단전환 25.56%.",
      "",
      "| 모델 | 단계 | 통과 | OfficialVal | 숫자×1.2 Flip | 문자×0.8 Flip | B8D 최악 | 16클래스 최악 |",
      "|---|---|---|---:|---:|---:|---:|---:|"]
    for r in ranked:
        summary.append("| "+r["variant"]+" | "+r["source"]+" | "+
         ("Yes" if r["passes_all_gates"] else "No")+" | "+
         " | ".join(f"{100*r[k]:.2f}%" for k in
           ("OfficialVal_mean","digits_1p2_flip","letters_0p8_flip","B8D_max_flip","all16_max_flip"))+" |")
    summary+=["","## 완료 판정"]
    if leader:
        summary += [f"- **동결 후보: {leader['variant']}** (기존 80-14 전처리 그대로 유지)",
           f"- OfficialVal {100*leader['OfficialVal_mean']:.4f}%, B8D 최악 {100*leader['B8D_max_flip']:.2f}%, 16클래스 최악 {100*leader['all16_max_flip']:.2f}%.",
           f"- 97.30% 목표: {'충족' if leader['OfficialVal_mean']>=.973 else '미달'}.",
           "- 팀원 손글씨 최종 Validation 전까지 배포/메인 반영 금지."]
    else:
        summary += ["- **사전등록된 Gate 전부 통과 조건 없음**. 최종 모델 채택·동결 금지.",
            f"- 최상위 차선책은 {fallback['variant'] if fallback else '없음'}이나 기준 미달이며 최종 후보로 선언하지 않는다."]
    (out/"SUMMARY.md").write_text("\n".join(summary)+"\n",encoding="utf-8")
    print("EXP017_FINAL_AGGREGATE_OK",len(eligible),"leader",leader["variant"] if leader else None,flush=True)
if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--coarse-inputs",required=True,type=Path)
    p.add_argument("--fine-inputs",required=True,type=Path)
    p.add_argument("--coarse-json",required=True,type=Path)
    p.add_argument("--out",required=True,type=Path)
    a=p.parse_args()
    run(a.coarse_inputs,a.fine_inputs,a.coarse_json,a.out)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")